<a href="https://colab.research.google.com/github/saadoonhammad/ieeecoins_data_imputation/blob/main/IEEE_COINS_Bi_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -U kaleido

# c01m045e01_2021

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import os
import random
import tensorflow as tf
import plotly.express as px
import plotly.io as pio
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# --- Reproducibility ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# --- Read Data ---
path = '/path/to/your/input_data/c01m045e01_2021.csv'
fnam = path[-19:-4]

df = pd.read_csv(path)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.set_index("timestamp", inplace=True)

scaler = MinMaxScaler()
df["scaled_temp"] = scaler.fit_transform(df[["temp_value_imp"]])

joblib.dump(scaler, "scaler.save")

train_df = df["2021-06-01":"2021-08-31"]
test_df = df["2021-09-01":"2021-09-30"]

# --- Functions ---
def create_sequences(series, T, N):
    X, y = [], []
    values = series.values
    for i in range(len(values) - T - N):
        X.append(values[i:i+T])
        y.append(values[i+T:i+T+N])
    return np.array(X), np.array(y)

def build_deep_bilstm_model(T, N, lr=0.0005):
    model = Sequential([
        Bidirectional(LSTM(64, return_sequences=True), input_shape=(T, 1)),
        Dropout(0.3),
        Bidirectional(LSTM(32, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(32)),
        Dense(N)
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return model

def impute_block_recursive(series, model, start_index, block_size, T, N):
    values = series.values.copy()
    current_index = start_index
    while current_index < start_index + block_size:
        context = values[current_index - T:current_index]
        if len(context) != T or np.isnan(context).any():
            current_index += N
            continue
        input_seq = context.reshape(1, T, 1)
        pred = model.predict(input_seq, verbose=0).flatten()
        for i in range(N):
            if current_index + i < len(values):
                values[current_index + i] = pred[i]
        current_index += N
    return pd.Series(values, index=series.index)

def evaluate_metrics(true, pred, start, length):
    mask = ~np.isnan(true[start:start+length])
    return (
        mean_absolute_error(true[start:start+length][mask], pred[start:start+length][mask]),
        root_mean_squared_error(true[start:start+length][mask], pred[start:start+length][mask]),
        mean_absolute_percentage_error(true[start:start+length][mask], pred[start:start+length][mask])
    )

def plot_imputation_plotly(true_series, missing_series, imputed_series, gap_start, gap_end, gap_size):
    df_plot = pd.DataFrame({
        "timestamp": true_series.index,
        "Original": true_series.values,
        "With Missing": missing_series.values,
        "Imputed": imputed_series.values
    })
    df_long = df_plot.melt(id_vars="timestamp", var_name="Type", value_name="Temperature")
    fig = px.line(df_long, x="timestamp", y="Temperature", color="Type", title=f"Gap Size {gap_size}")
    fig.add_vrect(x0=true_series.index[gap_start], x1=true_series.index[gap_end-1],
                  fillcolor="lightgray", opacity=0.3, line_width=0)
    fig.update_layout(width=1100, height=500, template="plotly_white")
    pio.write_image(fig, f"plots_bilstm/{fnam}_gap_{gap_size}.png")
    fig.show()

# --- Train Final Best BiLSTM ---
T, N = 144, 72
X_train, y_train = create_sequences(train_df["scaled_temp"], T, N)
X_train = X_train.reshape(-1, T, 1)
y_train = y_train.reshape(-1, N)

model = build_deep_bilstm_model(T, N)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model.fit(X_train, y_train, epochs=65, batch_size=64, validation_split=0.2, shuffle=False, callbacks=[early_stop], verbose=1)

model.save(f"{fnam}_final_bilstm_model.h5")

# --- Evaluate on All Gaps ---
os.makedirs("plots_bilstm", exist_ok=True)

gap_sizes = [150, 300, 450, 600]
results = []

for gap_size in gap_sizes:
    gap_start = 500
    test_missing = test_df.copy()
    test_missing.iloc[gap_start:gap_start+gap_size] = np.nan

    imputed_series = impute_block_recursive(test_missing["scaled_temp"], model, gap_start, gap_size, T, N)

    true_vals = scaler.inverse_transform(test_df["scaled_temp"].values.reshape(-1, 1)).flatten()
    imputed_vals = scaler.inverse_transform(imputed_series.values.reshape(-1, 1)).flatten()
    missing_vals = test_missing["temp_value_imp"].values

    s_true = pd.Series(true_vals, index=test_df.index)
    s_pred = pd.Series(imputed_vals, index=test_df.index)
    s_missing = pd.Series(missing_vals, index=test_df.index)

    mae, rmse, mape = evaluate_metrics(s_true, s_pred, gap_start, gap_size)
    results.append({
        "Gap_Size": gap_size,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape * 100
    })

    plot_imputation_plotly(s_true, s_missing, s_pred, gap_start, gap_start + gap_size, gap_size)

    comparison_df = pd.DataFrame({
        "timestamp": s_true.index,
        "original": s_true.values,
        "with_missing": s_missing.values,
        "imputed": s_pred.values
    })
    comparison_df.to_csv(f"{fnam}_gap_{gap_size}_bilstm.csv", index=False)

# --- Final Results Summary
results_df = pd.DataFrame(results)
results_df.to_csv(f"{fnam}_bilstm_final_results.csv", index=False)
print(results_df)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 91s 482ms/step - loss: 0.1190 - val_loss: 0.0142
Epoch 2/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 72s 443ms/step - loss: 0.0162 - val_loss: 0.0135
Epoch 3/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 86s 465ms/step - loss: 0.0155 - val_loss: 0.0131
Epoch 4/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 77s 436ms/step - loss: 0.0148 - val_loss: 0.0127
Epoch 5/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 71s 436ms/step - loss: 0.0141 - val_loss: 0.0124
Epoch 6/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 83s 443ms/step - loss: 0.0134 - val_loss: 0.0123
Epoch 7/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 83s 452ms/step - loss: 0.0130 - val_loss: 0.0118
Epoch 8/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 80s 442ms/step - loss: 0.0123 - val_loss: 0.0113
Epoch 9/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 72s 442ms/step - loss: 0.0115 - val_loss: 0.0107
Epoch 10/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 82s 443ms/step - loss: 0.0107 - val_loss: 0.0101
Epoch 11/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 82s 441ms/step - loss: 0.0098 - val_loss: 0.0102
Epoch 12/65
163/163

   Gap_Size       MAE      RMSE  MAPE (%)
0       150  1.044291  1.480245  5.816203
1       300  1.180146  1.606429  6.075532
2       450  1.554605  1.962709  7.375092
3       600  1.607167  2.046953  7.706272


# c05m105e08_2021

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import os
import random
import tensorflow as tf
import plotly.express as px
import plotly.io as pio
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# --- Reproducibility ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# --- Read Data ---
path = '/path/to/your/input_data/c05m105e08_2021.csv'
fnam = path[-19:-4]

df = pd.read_csv(path)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.set_index("timestamp", inplace=True)

scaler = MinMaxScaler()
df["scaled_temp"] = scaler.fit_transform(df[["temp_value_imp"]])

joblib.dump(scaler, "scaler.save")

train_df = df["2021-06-01":"2021-08-31"]
test_df = df["2021-09-01":"2021-09-30"]

# --- Functions ---
def create_sequences(series, T, N):
    X, y = [], []
    values = series.values
    for i in range(len(values) - T - N):
        X.append(values[i:i+T])
        y.append(values[i+T:i+T+N])
    return np.array(X), np.array(y)

def build_deep_bilstm_model(T, N, lr=0.0005):
    model = Sequential([
        Bidirectional(LSTM(128, return_sequences=True), input_shape=(T, 1)),
        Dropout(0.2),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.2),
        Bidirectional(LSTM(32)),
        Dense(N)
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return model

def impute_block_recursive(series, model, start_index, block_size, T, N):
    values = series.values.copy()
    current_index = start_index
    while current_index < start_index + block_size:
        context = values[current_index - T:current_index]
        if len(context) != T or np.isnan(context).any():
            current_index += N
            continue
        input_seq = context.reshape(1, T, 1)
        pred = model.predict(input_seq, verbose=0).flatten()
        for i in range(N):
            if current_index + i < len(values):
                values[current_index + i] = pred[i]
        current_index += N
    return pd.Series(values, index=series.index)

def evaluate_metrics(true, pred, start, length):
    mask = ~np.isnan(true[start:start+length])
    return (
        mean_absolute_error(true[start:start+length][mask], pred[start:start+length][mask]),
        root_mean_squared_error(true[start:start+length][mask], pred[start:start+length][mask]),
        mean_absolute_percentage_error(true[start:start+length][mask], pred[start:start+length][mask])
    )

def plot_imputation_plotly(true_series, missing_series, imputed_series, gap_start, gap_end, gap_size):
    df_plot = pd.DataFrame({
        "timestamp": true_series.index,
        "Original": true_series.values,
        "With Missing": missing_series.values,
        "Imputed": imputed_series.values
    })
    df_long = df_plot.melt(id_vars="timestamp", var_name="Type", value_name="Temperature")
    fig = px.line(df_long, x="timestamp", y="Temperature", color="Type", title=f"Gap Size {gap_size}")
    fig.add_vrect(x0=true_series.index[gap_start], x1=true_series.index[gap_end-1],
                  fillcolor="lightgray", opacity=0.3, line_width=0)
    fig.update_layout(width=1100, height=500, template="plotly_white")
    pio.write_image(fig, f"plots_bilstm/{fnam}_gap_{gap_size}.png")
    fig.show()

# --- Train Final Best BiLSTM ---
T, N = 144, 72
X_train, y_train = create_sequences(train_df["scaled_temp"], T, N)
X_train = X_train.reshape(-1, T, 1)
y_train = y_train.reshape(-1, N)

model = build_deep_bilstm_model(T, N)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model.fit(X_train, y_train, epochs=65, batch_size=64, validation_split=0.2, shuffle=False, callbacks=[early_stop], verbose=1)

model.save(f"{fnam}_final_bilstm_model.h5")

# --- Evaluate on All Gaps ---
os.makedirs("plots_bilstm", exist_ok=True)

gap_sizes = [150, 300, 450, 600]
results = []

for gap_size in gap_sizes:
    gap_start = 500
    test_missing = test_df.copy()
    test_missing.iloc[gap_start:gap_start+gap_size] = np.nan

    imputed_series = impute_block_recursive(test_missing["scaled_temp"], model, gap_start, gap_size, T, N)

    true_vals = scaler.inverse_transform(test_df["scaled_temp"].values.reshape(-1, 1)).flatten()
    imputed_vals = scaler.inverse_transform(imputed_series.values.reshape(-1, 1)).flatten()
    missing_vals = test_missing["temp_value_imp"].values

    s_true = pd.Series(true_vals, index=test_df.index)
    s_pred = pd.Series(imputed_vals, index=test_df.index)
    s_missing = pd.Series(missing_vals, index=test_df.index)

    mae, rmse, mape = evaluate_metrics(s_true, s_pred, gap_start, gap_size)
    results.append({
        "Gap_Size": gap_size,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape * 100
    })

    plot_imputation_plotly(s_true, s_missing, s_pred, gap_start, gap_start + gap_size, gap_size)

    comparison_df = pd.DataFrame({
        "timestamp": s_true.index,
        "original": s_true.values,
        "with_missing": s_missing.values,
        "imputed": s_pred.values
    })
    comparison_df.to_csv(f"{fnam}_gap_{gap_size}_bilstm.csv", index=False)

# --- Final Results Summary
results_df = pd.DataFrame(results)
results_df.to_csv(f"{fnam}_bilstm_final_results.csv", index=False)
print(results_df)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 149s 856ms/step - loss: 0.0900 - val_loss: 0.0133
Epoch 2/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 145s 876ms/step - loss: 0.0183 - val_loss: 0.0129
Epoch 3/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 197s 847ms/step - loss: 0.0171 - val_loss: 0.0127
Epoch 4/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 144s 881ms/step - loss: 0.0162 - val_loss: 0.0127
Epoch 5/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 145s 881ms/step - loss: 0.0157 - val_loss: 0.0127
Epoch 6/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 139s 854ms/step - loss: 0.0153 - val_loss: 0.0127
Epoch 7/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 139s 852ms/step - loss: 0.0150 - val_loss: 0.0127
Epoch 8/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 145s 871ms/step - loss: 0.0147 - val_loss: 0.0126
Epoch 9/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 202s 870ms/step - loss: 0.0144 - val_loss: 0.0125
Epoch 10/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 141s 864ms/step - loss: 0.0143 - val_loss: 0.0124
Epoch 11/65
163/163 ━━━━━━━━━━━━━━━━━━━━ 146s 889ms/step - loss: 0.0141 - val_loss: 0.0122
Epoch 12

   Gap_Size       MAE      RMSE  MAPE (%)
0       150  1.202842  1.447665  6.068242
1       300  1.424614  1.802752  7.066863
2       450  1.741758  2.131571  8.473278
3       600  2.071778  2.473201  9.992907


# c05m124e01_2021

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import os
import random
import tensorflow as tf
import plotly.express as px
import plotly.io as pio
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# --- Reproducibility ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# --- Read Data ---
path = '/path/to/your/input_data/c05m124e01_2021.csv'
fnam = path[-19:-4]

df = pd.read_csv(path)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.set_index("timestamp", inplace=True)

scaler = MinMaxScaler()
df["scaled_temp"] = scaler.fit_transform(df[["temp_value_imp"]])

joblib.dump(scaler, "scaler.save")

train_df = df["2021-06-01":"2021-08-31"]
test_df = df["2021-09-01":"2021-09-30"]

# --- Functions ---
def create_sequences(series, T, N):
    X, y = [], []
    values = series.values
    for i in range(len(values) - T - N):
        X.append(values[i:i+T])
        y.append(values[i+T:i+T+N])
    return np.array(X), np.array(y)

def build_deep_bilstm_model(T, N, lr=0.0005):
    model = Sequential([
        Bidirectional(LSTM(128, return_sequences=True), input_shape=(T, 1)),
        Dropout(0.2),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.2),
        Bidirectional(LSTM(32)),
        Dense(N)
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return model

def impute_block_recursive(series, model, start_index, block_size, T, N):
    values = series.values.copy()
    current_index = start_index
    while current_index < start_index + block_size:
        context = values[current_index - T:current_index]
        if len(context) != T or np.isnan(context).any():
            current_index += N
            continue
        input_seq = context.reshape(1, T, 1)
        pred = model.predict(input_seq, verbose=0).flatten()
        for i in range(N):
            if current_index + i < len(values):
                values[current_index + i] = pred[i]
        current_index += N
    return pd.Series(values, index=series.index)

def evaluate_metrics(true, pred, start, length):
    mask = ~np.isnan(true[start:start+length])
    return (
        mean_absolute_error(true[start:start+length][mask], pred[start:start+length][mask]),
        root_mean_squared_error(true[start:start+length][mask], pred[start:start+length][mask]),
        mean_absolute_percentage_error(true[start:start+length][mask], pred[start:start+length][mask])
    )

def plot_imputation_plotly(true_series, missing_series, imputed_series, gap_start, gap_end, gap_size):
    df_plot = pd.DataFrame({
        "timestamp": true_series.index,
        "Original": true_series.values,
        "With Missing": missing_series.values,
        "Imputed": imputed_series.values
    })
    df_long = df_plot.melt(id_vars="timestamp", var_name="Type", value_name="Temperature")
    fig = px.line(df_long, x="timestamp", y="Temperature", color="Type", title=f"Gap Size {gap_size}")
    fig.add_vrect(x0=true_series.index[gap_start], x1=true_series.index[gap_end-1],
                  fillcolor="lightgray", opacity=0.3, line_width=0)
    fig.update_layout(width=1100, height=500, template="plotly_white")
    pio.write_image(fig, f"plots_bilstm/{fnam}_gap_{gap_size}.png")
    fig.show()

# --- Train Final Best BiLSTM ---
T, N = 144, 72
X_train, y_train = create_sequences(train_df["scaled_temp"], T, N)
X_train = X_train.reshape(-1, T, 1)
y_train = y_train.reshape(-1, N)

model = build_deep_bilstm_model(T, N)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model.fit(X_train, y_train, epochs=65, batch_size=64, validation_split=0.2, shuffle=False, callbacks=[early_stop], verbose=1)

model.save(f"{fnam}_final_bilstm_model.h5")

# --- Evaluate on All Gaps ---
os.makedirs("plots_bilstm", exist_ok=True)

gap_sizes = [150, 300, 450, 600]
results = []

for gap_size in gap_sizes:
    gap_start = 500
    test_missing = test_df.copy()
    test_missing.iloc[gap_start:gap_start+gap_size] = np.nan

    imputed_series = impute_block_recursive(test_missing["scaled_temp"], model, gap_start, gap_size, T, N)

    true_vals = scaler.inverse_transform(test_df["scaled_temp"].values.reshape(-1, 1)).flatten()
    imputed_vals = scaler.inverse_transform(imputed_series.values.reshape(-1, 1)).flatten()
    missing_vals = test_missing["temp_value_imp"].values

    s_true = pd.Series(true_vals, index=test_df.index)
    s_pred = pd.Series(imputed_vals, index=test_df.index)
    s_missing = pd.Series(missing_vals, index=test_df.index)

    mae, rmse, mape = evaluate_metrics(s_true, s_pred, gap_start, gap_size)
    results.append({
        "Gap_Size": gap_size,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape * 100
    })

    plot_imputation_plotly(s_true, s_missing, s_pred, gap_start, gap_start + gap_size, gap_size)

    comparison_df = pd.DataFrame({
        "timestamp": s_true.index,
        "original": s_true.values,
        "with_missing": s_missing.values,
        "imputed": s_pred.values
    })
    comparison_df.to_csv(f"{fnam}_gap_{gap_size}_bilstm.csv", index=False)

# --- Final Results Summary
results_df = pd.DataFrame(results)
results_df.to_csv(f"{fnam}_bilstm_final_results.csv", index=False)
print(results_df)